### DRL-DRL fitness under outcome restrictions
Token-based replay fitness of DRL-DRL's simulated logs against an Inductive Miner model of the training log (same method as `token_replay_fitness.ipynb`).
Restricted runs are also split into **restricted** cases (the rules applied) and **free** cases (`restricted_cases/run_XX.csv`).

In [ ]:
from pathlib import Path

import pandas as pd
import pm4py
from scipy import stats

NOISE = 0.2
POLICY = "DRL-DRL"
SCENARIOS = {  # scenario -> log
    "BPIC17_R_validate_after_call_offers": "BPIC_2017",
    "BPIC12_R1_no_end_after_leads": "BPIC_2012",
    "AC_R_limit_cancel": "AcademicCredentials",
}
DATA = Path("../../data")
RESTRICTED = DATA / "evaluation_results_restricted"
UNRESTRICTED = DATA / "evaluation_results/flex"


def load(path, cases=None):
    df = pd.read_csv(path, usecols=["case_id", "activity", "end_time"])
    df["case_id"] = df["case_id"].astype(str)
    if cases is not None:
        df = df[df["case_id"].isin(cases)]
    df["activity"] = df["activity"].astype(str).str.strip()
    df["end_time"] = pd.to_datetime(df["end_time"], format="mixed")
    return pm4py.format_dataframe(df, case_id="case_id", activity_key="activity", timestamp_key="end_time")


def fitness(log, model):
    return pm4py.fitness_token_based_replay(log, *model)["log_fitness"]


def ci95(v):
    return stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / len(v) ** 0.5 if len(v) > 1 else 0.0

In [ ]:
rows = []
for scen, log in SCENARIOS.items():
    train = load(DATA / f"logs/{log}/{log}_train.csv")
    model = pm4py.discover_petri_net_inductive(train, noise_threshold=NOISE)
    rows.append({"scenario": scen, "env": "train", "run": 0, "all": fitness(train, model)})

    for env, base in (("unrestricted", UNRESTRICTED), ("restricted", RESTRICTED / scen)):
        for k, run in enumerate(sorted((base / log / POLICY / "simulated_logs").glob("sim_run_*.csv")), 1):
            row = {"scenario": scen, "env": env, "run": k, "all": fitness(load(run), model)}
            if env == "restricted":
                ids = set(pd.read_csv(base / log / POLICY / f"restricted_cases/run_{k:02d}.csv")["case_id"].astype(str))
                all_ids = set(pd.read_csv(run, usecols=["case_id"])["case_id"].astype(str))
                row["restricted"] = fitness(load(run, ids), model)
                row["free"] = fitness(load(run, all_ids - ids), model)
            rows.append(row)
    print(f"[{scen}] done")

runs = pd.DataFrame(rows)

In [ ]:
agg = runs.groupby(["scenario", "env"], sort=False)[["all", "restricted", "free"]].agg(["mean", ci95]).round(4)
agg.columns = ["_".join(c) for c in agg.columns]

#Path("results").mkdir(exist_ok=True)
#runs.to_csv("results/restricted_drl_fitness_runs.csv", index=False)
#agg.to_csv("results/restricted_drl_fitness_aggregated.csv")
agg